# Unidad 5 · Colab 3 de 3
## Rate limiting, ética/legal y proyecto de competitive intelligence

**Objetivos de este notebook**

- Implementar rate limiting y rotación de user-agents para scrapear de forma responsable.
- Verificar `robots.txt` antes de scrapear un sitio.
- Entender los aspectos éticos y legales del web scraping (sin que esto reemplace asesoramiento legal profesional).
- Aplicar todo lo visto en un pipeline de **competitive intelligence**: extraer precios y productos de la competencia y guardarlos en una base de datos con historial.

> **Nivel:** intermedio. Este notebook conecta con los Colab 1 y 2 (extracción) y con la Unidad 6 (para programar la recolección periódica).

---

## 1. Rate limiting

Hacer muchos requests muy rápido puede saturar el servidor, disparar bloqueos (HTTP 429) o directamente afectar la disponibilidad del sitio. Buenas prácticas:

- Esperar un intervalo fijo (`time.sleep`) entre requests.
- Respetar el header `Retry-After` si el servidor responde `429 Too Many Requests`.
- Usar **backoff exponencial**: si falla, esperar cada vez más antes de reintentar.

```python
import time
import requests

def get_con_backoff(url, headers, intentos=5):
    espera = 1
    for _ in range(intentos):
        resp = requests.get(url, headers=headers, timeout=10)
        if resp.status_code == 429:
            time.sleep(int(resp.headers.get('Retry-After', espera)))
            espera *= 2
            continue
        resp.raise_for_status()
        return resp
    raise RuntimeError('No se pudo obtener la pagina tras varios intentos')
```

### Ejercicio 1 — Rate limiting entre páginas

Modificá el loop de paginación del Colab 1 (recorrer `quotes.toscrape.com`) para que, además de esperar 1 segundo fijo entre páginas, use `get_con_backoff` en vez de `requests.get` directo.

<details>
<summary>💡 Ver solución</summary>

```python
todas = []
pagina = 1
headers = {'User-Agent': 'Mozilla/5.0 (compatible; CursoScrapingBot/1.0)'}

while True:
    url = f'http://quotes.toscrape.com/page/{pagina}/'
    resp = get_con_backoff(url, headers)
    soup = BeautifulSoup(resp.text, 'html.parser')
    for q in soup.select('.quote'):
        todas.append({
            'cita': q.select_one('.text').get_text(strip=True),
            'autor': q.select_one('.author').get_text(strip=True),
        })
    if soup.select_one('li.next') is None:
        break
    pagina += 1
    time.sleep(1)
```

</details>

## 2. Rotación de user-agents

Algunos sitios bloquean o limitan requests que llegan siempre con el mismo `User-Agent`. Rotar entre varios (reales, de navegadores comunes) reduce ese riesgo — aunque **no** reemplaza respetar el rate limiting ni los términos del sitio.

```python
import random

USER_AGENTS = [
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/124.0 Safari/537.36',
    'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 Safari/605.1.15',
    'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 Chrome/124.0 Safari/537.36',
]

def headers_aleatorios():
    return {'User-Agent': random.choice(USER_AGENTS)}
```

También existe la librería [fake-useragent](https://pypi.org/project/fake-useragent/) para generar user-agents realistas y actualizados automáticamente.

### Ejercicio 2 — Sesión con rotación de user-agent

Escribí una función `request_rotado(url)` que use `headers_aleatorios()` en cada llamada y devuelva la respuesta (usando `get_con_backoff`).

<details>
<summary>💡 Ver solución</summary>

```python
def request_rotado(url):
    return get_con_backoff(url, headers_aleatorios())
```

</details>

## 3. `robots.txt`

Es un archivo público (`https://sitio.com/robots.txt`) donde el sitio indica qué rutas no quiere que los bots recorran. No es una barrera técnica (no te va a bloquear el request), pero **respetarlo es la norma ética y, en muchos casos, contractual** (suele estar referenciado en los Términos de Servicio).

```python
from urllib.robotparser import RobotFileParser

rp = RobotFileParser()
rp.set_url('http://quotes.toscrape.com/robots.txt')
rp.read()

print(rp.can_fetch('*', 'http://quotes.toscrape.com/page/2/'))
```

Documentación oficial: [robotparser (Python stdlib)](https://docs.python.org/3/library/urllib.robotparser.html) · [Introducción a robots.txt (Google)](https://developers.google.com/search/docs/crawling-indexing/robots/intro)

### Ejercicio 3 — Chequear robots.txt antes de scrapear

Escribí una función `puedo_scrapear(url)` que devuelva `True`/`False` verificando el `robots.txt` del dominio correspondiente para esa URL, usando tu propio user-agent.

<details>
<summary>💡 Ver solución</summary>

```python
from urllib.parse import urlparse

def puedo_scrapear(url):
    partes = urlparse(url)
    base = f'{partes.scheme}://{partes.netloc}'
    rp = RobotFileParser()
    rp.set_url(f'{base}/robots.txt')
    rp.read()
    return rp.can_fetch('CursoScrapingBot', url)

puedo_scrapear('http://quotes.toscrape.com/page/2/')
```

</details>

## 4. Ética y aspectos legales del scraping

> Esto es una introducción general, **no es asesoramiento legal**. La legislación varía según el país y el caso concreto; ante dudas reales sobre un proyecto específico, consultá con un abogado.

Puntos a tener en cuenta:

- **Términos de Servicio (ToS):** muchos sitios prohíben explícitamente el scraping automatizado. Violarlos puede derivar en un reclamo por incumplimiento de contrato, incluso si el dato en sí es público.
- **Datos personales:** si vas a extraer información que identifica personas, aplican regulaciones de protección de datos (por ejemplo, el RGPD/GDPR en la Unión Europea), independientemente de que el dato sea de acceso público.
- **Propiedad intelectual y derecho de bases de datos:** el contenido (textos, imágenes, catálogos) puede estar protegido; reutilizarlo o republicarlo tiene implicancias distintas a simplemente analizarlo de forma interna.
- **Antecedente relevante:** el caso [hiQ Labs v. LinkedIn](https://en.wikipedia.org/wiki/HiQ_Labs_v._LinkedIn) mostró que, si bien los tribunales de apelación en EE.UU. consideraron que scrapear datos públicos no viola por sí solo las leyes de acceso no autorizado a sistemas informáticos, el caso terminó en un acuerdo (2022) donde hiQ se comprometió a dejar de scrapear LinkedIn y a eliminar los datos obtenidos, por incumplir los Términos de Servicio del sitio. La lección: dato público no equivale a dato sin restricciones contractuales.

Buenas prácticas éticas, más allá de lo estrictamente legal:

- Preferir una API oficial si el sitio la ofrece, en vez de scrapear.
- Identificarte con un User-Agent honesto cuando sea posible.
- No sobrecargar el servidor (rate limiting).
- No extraer ni almacenar datos personales sin una base legal para hacerlo.
- Usar los datos extraídos solo para el fin declarado (en este caso, un ejercicio educativo o un análisis interno de mercado).

### Checklist antes de scrapear un sitio real

- [ ] Revisé el `robots.txt` del sitio
- [ ] Leí (al menos) la sección de Términos de Servicio sobre scraping/bots
- [ ] Definí un rate limit razonable y un User-Agent identificable
- [ ] Confirmé que no voy a extraer datos personales sin base legal
- [ ] Tengo un propósito legítimo y documentado para los datos extraídos

## 5. Aplicación práctica: competitive intelligence

El caso de uso de esta unidad: extraer precios y productos de sitios de competidores de forma periódica, y guardarlos en una base de datos con historial, para poder analizar variaciones de precio en el tiempo. Lo vamos a construir sobre [books.toscrape.com](http://books.toscrape.com) (sitio de práctica) simulando un competidor.

## 6. Diseño de la base de datos

Para un histórico de precios necesitamos, como mínimo, una tabla de productos y una tabla de observaciones de precio en el tiempo (no pisamos el precio anterior, lo agregamos como una fila nueva):

```sql
CREATE TABLE producto (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    url TEXT UNIQUE NOT NULL,
    titulo TEXT NOT NULL
);

CREATE TABLE precio_observado (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    producto_id INTEGER NOT NULL REFERENCES producto(id),
    precio REAL NOT NULL,
    disponibilidad TEXT,
    fecha_scrape TEXT NOT NULL
);
```

### Ejercicio 4 — Justificar el modelo

¿Por qué conviene una tabla `precio_observado` separada de `producto`, en vez de una sola tabla con una columna `precio` que se actualiza cada vez que se vuelve a scrapear?

<details>
<summary>💡 Ver solución</summary>

Si actualizáramos el precio en la misma fila, perderíamos el historial: no podríamos saber cómo varió el precio en el tiempo. Con una tabla separada, cada scrape agrega una fila nueva con su `fecha_scrape`, y podés reconstruir la evolución completa del precio de cada producto — útil para detectar bajadas, promociones o tendencias de la competencia.

</details>

In [ ]:
import sqlite3

conn = sqlite3.connect('competidores.db')
cur = conn.cursor()

cur.execute('''
CREATE TABLE IF NOT EXISTS producto (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    url TEXT UNIQUE NOT NULL,
    titulo TEXT NOT NULL
)
''')

cur.execute('''
CREATE TABLE IF NOT EXISTS precio_observado (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    producto_id INTEGER NOT NULL REFERENCES producto(id),
    precio REAL NOT NULL,
    disponibilidad TEXT,
    fecha_scrape TEXT NOT NULL
)
''')
conn.commit()
print('Base de datos lista')

### Ejercicio 5 — Pipeline completo: scrapear y guardar

Completá la función `guardar_producto(conn, url, titulo, precio, disponibilidad)` para que: (1) inserte el producto en `producto` si no existe (`INSERT OR IGNORE`), (2) obtenga su `id`, y (3) inserte una fila en `precio_observado` con la fecha actual.

In [ ]:
from datetime import datetime

def guardar_producto(conn, url, titulo, precio, disponibilidad):
    cur = conn.cursor()
    # TODO: INSERT OR IGNORE en producto, luego SELECT id, luego INSERT en precio_observado
    pass

<details>
<summary>💡 Ver solución</summary>

```python
from datetime import datetime

def guardar_producto(conn, url, titulo, precio, disponibilidad):
    cur = conn.cursor()
    cur.execute('INSERT OR IGNORE INTO producto (url, titulo) VALUES (?, ?)', (url, titulo))
    cur.execute('SELECT id FROM producto WHERE url = ?', (url,))
    producto_id = cur.fetchone()[0]
    cur.execute(
        'INSERT INTO precio_observado (producto_id, precio, disponibilidad, fecha_scrape) VALUES (?, ?, ?, ?)',
        (producto_id, precio, disponibilidad, datetime.utcnow().isoformat())
    )
    conn.commit()
```

</details>

In [ ]:
import re

resp = get_con_backoff('http://books.toscrape.com/', headers_aleatorios())
soup = BeautifulSoup(resp.text, 'html.parser')

for libro in soup.select('article.product_pod'):
    titulo = libro.h3.a['title']
    url = libro.h3.a['href']
    precio_texto = libro.select_one('.price_color').get_text(strip=True)
    precio = float(re.sub(r'[^0-9.]', '', precio_texto))
    disponibilidad = libro.select_one('.availability').get_text(strip=True)
    guardar_producto(conn, url, titulo, precio, disponibilidad)

cur.execute('SELECT COUNT(*) FROM producto')
print('Productos guardados:', cur.fetchone()[0])

## 7. Programar la recolección periódica

Para que sea competitive intelligence de verdad, el scraping tiene que correr solo, con regularidad. Dos formas simples, conectando con la Unidad 6:

- Un **workflow de GitHub Actions programado** (`on: schedule`, sintaxis cron) que corra el script y guarde los datos.
- Un job programado en la plataforma PaaS (Render/Railway ofrecen Cron Jobs).

```yaml
on:
  schedule:
    - cron: '0 6 * * *'   # todos los dias a las 06:00 UTC
```

Documentación oficial: [Eventos de workflow: schedule](https://docs.github.com/es/actions/writing-workflows/choosing-when-your-workflow-runs/events-that-trigger-workflows#schedule)

## Mini-proyecto final integrador

1. Elegí un sitio de práctica (o uno propio donde tengas permiso) con listado de productos y precios.
2. Verificá su `robots.txt` con `puedo_scrapear()`.
3. Armá el pipeline completo: request con rate limiting + user-agent rotado → parseo con BeautifulSoup (o Playwright si es dinámico) → guardado en SQLite con historial de precios.
4. Corré el pipeline dos veces (simulando dos días distintos) y consultá la tabla `precio_observado` para ver la evolución.
5. (Opcional) Escribí el workflow de GitHub Actions para programarlo.

**Entregable:** script/notebook del pipeline + archivo `competidores.db` con al menos dos observaciones de precio por producto + una consulta SQL que muestre la variación de precio de un producto entre ambas corridas.

## Autoevaluación

- [ ] Implementé rate limiting con backoff en mis requests
- [ ] Roté user-agents en las requests
- [ ] Verifiqué `robots.txt` antes de scrapear
- [ ] Puedo explicar al menos dos riesgos legales/éticos del scraping
- [ ] Diseñé una base de datos que preserva el historial de precios
- [ ] Mi pipeline extrae y guarda datos de punta a punta

---

**Fin de la Unidad 5.** Recorriste el ciclo completo: inspeccionar y extraer datos estáticos y dinámicos, hacerlo de forma responsable, y aplicarlo a un caso real de competitive intelligence.